# Housing Price Prediction: From Traditional ML to Fine-Tuned LLMs

This notebook demonstrates Week 6 concepts from the LLM Engineering course:
- Data curation and preprocessing
- Evaluation framework with W&B integration
- Traditional ML baselines (Linear Regression, Random Forest, XGBoost)
- Neural network with PyTorch
- Zero-shot LLM comparison
- Fine-tuning LLaMA 3.2 3B with QLoRA

**Dataset:** USA Real Estate (Kaggle)
**W&B Project:** `llama-3.2-housing-pricer`

## Step 0: Setup and Imports

In [1]:
# Uncomment to install dependencies
!uv pip install kagglehub pandas numpy scikit-learn xgboost torch litellm wandb tqdm

Using Python 3.12.12 environment at: /Users/davemathews/Desktop/Learn/llm_engineering/.venv
Audited 9 packages in 13ms


In [2]:
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm

from week7.community_contributions.davenjeru.housing_pricer.data import load_raw_data, filter_data, create_splits, PRICE_MIN, PRICE_MAX
from week7.community_contributions.davenjeru.housing_pricer import create_description, format_for_inference, format_for_finetuning
from week7.community_contributions.davenjeru.housing_pricer.models import (
    RandomPricer, MeanPricer, 
    train_linear_regression, train_random_forest, train_xgboost,
    train_neural_network, predict_neural_network,
    predict_with_llm
)
from week7.community_contributions.davenjeru.housing_pricer.evaluation import (
    init_wandb, log_model_results,
    log_training_step, create_comparison_table
)

## Step 1: Load and Explore Data

In [3]:
df_raw = load_raw_data()
print(f"Raw data: {len(df_raw):,} rows")
df_raw.head()

Loading data from: /Users/davemathews/.cache/kagglehub/datasets/ahmedshahriarsakib/usa-real-estate-dataset/versions/25/realtor-data.zip.csv
Raw data: 2,226,382 rows


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN


In [4]:
print("\nColumn info:")
print(df_raw.dtypes)
print(f"\nPrice range: ${df_raw['price'].min():,.0f} - ${df_raw['price'].max():,.0f}")


Column info:
brokered_by       float64
status             object
price             float64
bed               float64
bath              float64
acre_lot          float64
street            float64
city               object
state              object
zip_code          float64
house_size        float64
prev_sold_date     object
dtype: object

Price range: $0 - $2,147,483,600


## Step 2: Filter and Split Data

In [5]:
df_clean = filter_data(df_raw)
print(f"After filtering (${PRICE_MIN:,} - ${PRICE_MAX:,}): {len(df_clean):,} rows")

After filtering ($50,000 - $2,000,000): 1,407,117 rows


In [6]:
train_df, val_df, test_df = create_splits(df_clean, train_size=500, val_size=50, test_size=50)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 510, Val: 50, Test: 50


## Step 3: Create Text Descriptions

Convert structured features to natural language for LLM-based pricing.

In [7]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df['description'] = train_df.apply(create_description, axis=1)
val_df['description'] = val_df.apply(create_description, axis=1)
test_df['description'] = test_df.apply(create_description, axis=1)

In [8]:
print("Example description:")
print(train_df['description'].iloc[0])
print(f"\nActual price: ${train_df['price'].iloc[0]:,.0f}")

Example description:
A 3 bedroom, 2.0 bathroom house with 1,401 square feet on a 0.18 acre lot located in Montgomery, Alabama, 36116.

Actual price: $169,900


## Step 4: Initialize W&B

All experiments will be logged to the `llama-3.2-housing-pricer` project.

In [9]:
import wandb
wandb.login()

wandb: Currently logged in as: davenmathews (davenjeru) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [10]:
init_wandb(
    config={
        "dataset": "usa-real-estate",
        "train_size": len(train_df),
        "val_size": len(val_df),
        "test_size": len(test_df),
        "price_range": f"${PRICE_MIN:,}-${PRICE_MAX:,}"
    },
    run_name="house-pricer-comparison"
)

wandb: setting up run m6pz2p73
wandb: Tracking run with wandb version 0.22.3
wandb: Run data is saved locally in /Users/davemathews/Desktop/Learn/llm_engineering/week6/community-contributions/davenjeru/wandb/run-20260308_172719-m6pz2p73
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run house-pricer-comparison
wandb: ⭐️ View project at https://wandb.ai/davenjeru/llama-3.2-housing-pricer
wandb: 🚀 View run at https://wandb.ai/davenjeru/llama-3.2-housing-pricer/runs/m6pz2p73
wandb: Detected [litellm, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


## Step 5: Baseline Models

Start with simple baselines to establish a floor for comparison.

In [11]:
results = []
y_test = test_df['price'].values

In [12]:
# Random baseline
random_pricer = RandomPricer(PRICE_MIN, PRICE_MAX)
random_preds = random_pricer.predict(test_df)
metrics = log_model_results("Random", y_test, random_preds)
results.append({"name": "Random", "type": "Baseline", **metrics})
print(f"Random Baseline - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

Random Baseline - MAE: $681,210, RMSE: $838,647, MAPE: 249.8%


In [13]:
# Mean baseline
mean_pricer = MeanPricer()
mean_pricer.fit(train_df['price'].values)
mean_preds = mean_pricer.predict(test_df)
metrics = log_model_results("Mean", y_test, mean_preds)
results.append({"name": "Mean", "type": "Baseline", **metrics})
print(f"Mean Baseline - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

Mean Baseline - MAE: $224,589, RMSE: $288,343, MAPE: 76.5%


## Step 6: Traditional ML Models

Use structured features (bed, bath, sqft, acres) for traditional ML.

In [14]:
features = ['bed', 'bath', 'house_size', 'acre_lot']
X_train = train_df[features].fillna(0).values
X_val = val_df[features].fillna(0).values
X_test = test_df[features].fillna(0).values
y_train = train_df['price'].values
y_val = val_df['price'].values

In [15]:
# Linear Regression
lr_model = train_linear_regression(X_train, y_train)
lr_preds = lr_model.predict(X_test)
metrics = log_model_results("LinearRegression", y_test, lr_preds)
results.append({"name": "LinearRegression", "type": "Traditional ML", **metrics})
print(f"Linear Regression - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

Linear Regression - MAE: $174,737, RMSE: $232,439, MAPE: 55.3%


In [16]:
# Random Forest
rf_model = train_random_forest(X_train, y_train)
rf_preds = rf_model.predict(X_test)
metrics = log_model_results("RandomForest", y_test, rf_preds)
results.append({"name": "RandomForest", "type": "Traditional ML", **metrics})
print(f"Random Forest - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

Random Forest - MAE: $193,769, RMSE: $253,324, MAPE: 66.1%


In [17]:
# XGBoost
xgb_model = train_xgboost(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
metrics = log_model_results("XGBoost", y_test, xgb_preds)
results.append({"name": "XGBoost", "type": "Traditional ML", **metrics})
print(f"XGBoost - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

XGBoost - MAE: $198,529, RMSE: $267,581, MAPE: 67.8%


## Step 7: Neural Network

Train a simple MLP on the structured features.

In [18]:
nn_model = train_neural_network(
    X_train, y_train,
    input_dim=4,
    epochs=100,
    learning_rate=1e-3,
    log_fn=log_training_step
)

In [19]:
nn_preds = predict_neural_network(nn_model, X_test)
metrics = log_model_results("NeuralNet", y_test, nn_preds)
results.append({"name": "NeuralNet", "type": "Deep Learning", **metrics})
print(f"Neural Network - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

Neural Network - MAE: $294,397, RMSE: $393,726, MAPE: 59.0%


## Step 8: Zero-Shot LLM Comparison

Test frontier LLMs on price estimation without fine-tuning.

**Note:** This uses a sample of 10 test examples to manage API costs.

In [20]:
sample_test = test_df.head(10)
sample_y_test = sample_test['price'].values

In [21]:
# GPT-4o-mini
gpt_preds = []
for _, row in tqdm(sample_test.iterrows(), total=len(sample_test), desc="GPT-4o-mini"):
    messages = format_for_inference(row['description'])
    pred = predict_with_llm("gpt-4o-mini", messages)
    gpt_preds.append(pred)

metrics = log_model_results("GPT-4o-mini", sample_y_test, gpt_preds)
results.append({"name": "GPT-4o-mini (zero-shot)", "type": "LLM", **metrics})
print(f"GPT-4o-mini - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

GPT-4o-mini: 100%|██████████| 10/10 [00:18<00:00,  1.82s/it]


GPT-4o-mini - MAE: $133,530, RMSE: $240,712, MAPE: 25.3%


In [22]:
# Claude Haiku (optional - uncomment to run)
# claude_preds = []
# for _, row in tqdm(sample_test.iterrows(), total=len(sample_test), desc="Claude Haiku"):
#     messages = format_for_inference(row['description'])
#     pred = predict_with_llm("claude-3-haiku-20240307", messages)
#     claude_preds.append(pred)

# metrics = log_model_results("Claude-Haiku", sample_y_test, claude_preds)
# results.append({"name": "Claude-Haiku (zero-shot)", "type": "LLM", **metrics})
# print(f"Claude Haiku - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

## Step 9: Final Comparison

In [23]:
create_comparison_table(results)

In [24]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(f"{'Model':<30} {'Type':<15} {'MAE':>12} {'MAPE':>8}")
print("-"*60)
for r in sorted(results, key=lambda x: x['mae']):
    print(f"{r['name']:<30} {r['type']:<15} ${r['mae']:>10,.0f} {r['mape']:>7.1f}%")
print("="*60)


MODEL COMPARISON SUMMARY
Model                          Type                     MAE     MAPE
------------------------------------------------------------
GPT-4o-mini (zero-shot)        LLM             $   133,530    25.3%
LinearRegression               Traditional ML  $   174,737    55.3%
RandomForest                   Traditional ML  $   193,769    66.1%
XGBoost                        Traditional ML  $   198,529    67.8%
Mean                           Baseline        $   224,589    76.5%
NeuralNet                      Deep Learning   $   294,397    59.0%
Random                         Baseline        $   681,210   249.8%


In [25]:
wandb.finish()
print("\nW&B run finished. View your results at: https://wandb.ai/")

wandb: updating run metadata; uploading artifact run-m6pz2p73-model_comparison
wandb: uploading artifact run-m6pz2p73-model_comparison; uploading media/table/model_comparison_31_05b006fadabfb869b803.table.json; uploading wandb-summary.json
wandb: uploading artifact run-m6pz2p73-model_comparison; uploading wandb-summary.json; uploading config.yaml
wandb: uploading artifact run-m6pz2p73-model_comparison
wandb: uploading history steps 28-31, summary
wandb: 
wandb: Run history:
wandb:       GPT-4o-mini/mae ▁
wandb:      GPT-4o-mini/mape ▁
wandb:      GPT-4o-mini/rmse ▁
wandb:  LinearRegression/mae ▁
wandb: LinearRegression/mape ▁
wandb: LinearRegression/rmse ▁
wandb:              Mean/mae ▁
wandb:             Mean/mape ▁
wandb:             Mean/rmse ▁
wandb:         NeuralNet/mae ▁
wandb:                   +13 ...
wandb: 
wandb: Run summary:
wandb:       GPT-4o-mini/mae 133530
wandb:      GPT-4o-mini/mape 25.29569
wandb:      GPT-4o-mini/rmse 240712.32415
wandb:  LinearRegression/mae 17473


W&B run finished. View your results at: https://wandb.ai/


## Step 10: Fine-Tuning LLaMA 3.2 3B with QLoRA

**Note:** This section should be run in Google Colab with a T4 GPU.

Copy the cells below to a Colab notebook, or use the separate `finetune_colab.ipynb`.

In [26]:
# Save training data for fine-tuning
finetune_data = train_df.apply(format_for_finetuning, axis=1).tolist()

import json
with open('train_finetune.jsonl', 'w') as f:
    for item in finetune_data:
        f.write(json.dumps(item) + '\n')

print(f"Saved {len(finetune_data)} training examples to train_finetune.jsonl")

Saved 510 training examples to train_finetune.jsonl


In [27]:
# Preview a training example
print("Training example format:")
print(finetune_data[0]['text'][:500] + "...")

Training example format:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a real estate appraiser. Based on property details, estimate the market price.
You only respond with the price, no other text.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
What is the estimated price of this property?

A 3 bedroom, 2.0 bathroom house with 1,401 square feet on a 0.18 acre lot located in Montgomery, Alabama, 36116.<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
$169,900<|eot_id|>...


### Fine-Tuning Code (Run in Colab)

```python
# Install dependencies
!pip install transformers peft bitsandbytes accelerate trl datasets huggingface_hub wandb

# Login to HuggingFace and W&B
from huggingface_hub import login
login()  # You need access to meta-llama/Llama-3.2-3B-Instruct

import wandb
wandb.login()

# Load and configure model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model_name = "meta-llama/Llama-3.2-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Load training data
dataset = load_dataset('json', data_files='train_finetune.jsonl', split='train')

# Training config with W&B
wandb.init(project="llama-3.2-housing-pricer", name="llama-3.2-3b-qlora")

training_args = SFTConfig(
    output_dir="./llama-house-pricer",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="wandb",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
)

trainer.train()
model.save_pretrained("./llama-house-pricer-final")
wandb.finish()
```